# 01 - Dados: da coleta à base de treino

Todas as fontes são públicas e anônimas (AWS/GCS Open Data). Ver `RELATORIO.md` para as trocas de fonte e suas justificativas.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
fig = lambda n: display(Image(filename=str(ROOT / 'reports/figures' / n)))
met = lambda n: pd.read_csv(ROOT / 'reports/metrics' / n)

In [2]:
from src.common.config import load_config, data_path
cfg = load_config()
st = pd.read_parquet('data/interim/stations.parquet') if Path('data/interim/stations.parquet').exists() else pd.read_parquet('data/sample/stations.parquet')
print(len(st), 'estações CETESB com PM2.5'); st.head(40)

30 estações CETESB com PM2.5


,location_id,name,lat,lon
0,5222,Santa Gertrudes,-22.459955,-47.536298
1,5231,Taubaté,-23.032351,-45.575805
2,5233,Cid.Universitária-USP-Ipen,-23.566342,-46.737414
3,5239,Osasco,-23.526721,-46.792078
4,5243,Pinheiros,-23.561460,-46.702017
5,5252,Congonhas,-23.616320,-46.663466
6,5255,Grajaú-Parelheiros,-23.776266,-46.696961
7,5256,Piracicaba,-22.701222,-47.649653
8,5257,Campinas-V.União,-22.946728,-47.119281
9,5260,Jundiaí,-23.192004,-46.897097


In [3]:
df = pd.read_parquet('data/processed/station_days.parquet')
print(df.shape); print(df['date'].min().date(), '->', df['date'].max().date())
df.groupby('name').agg(dias=('pm25','size'), pm25_medio=('pm25','mean'), pm25_p95=('pm25', lambda s: s.quantile(.95))).sort_values('dias', ascending=False).round(1)

(11543, 58)
2021-01-13 -> 2023-04-04


,dias,pm25_medio,pm25_p95
name,,,
Jundiaí,582,11.7,24.9
Itaim Paulista,517,14.0,33.0
Marg.Tietê-Pte Remédios,512,16.8,36.2
Guaratinguetá,446,11.4,21.6
Congonhas,432,15.1,29.5
Parque D.Pedro II,432,14.5,31.8
Guarulhos-Pimentas,430,13.8,31.7
S.Bernardo-Centro,419,12.9,25.5
Santos-Ponta da Praia,414,10.8,20.4


## Falhas de AOD por nuvem e preenchimento espacial causal
Nível 0 = célula com retrieval no dia; 1 = janela 3x3; 2 = janela 9x9; 4 = sem AOD (NaN tratado pelo modelo). Nenhuma observação futura é usada.

In [4]:
lvl = df['aod_fill_level'].value_counts(normalize=True).sort_index().rename('fração'); display(lvl.to_frame())
df.assign(mes=df.date.dt.month).groupby('mes')['aod'].apply(lambda s: s.notna().mean()).rename('AOD observado na célula (fração)').to_frame().T.round(2)

,fração
aod_fill_level,
0,0.508100
1,0.111236
2,0.119813
4,0.260851


mes,1,2,3,4,5,6,7,8,9,10,11,12
AOD observado na célula (fração),0.34,0.38,0.58,0.58,0.67,0.55,0.88,0.63,0.62,0.42,0.51,0.39


## Coerência física: correlações com o PM2.5 medido

In [5]:
cols = ['aod','aod_filled','aod_dry_blh','no2_filled','blh_pm','log_vent','rh','tp','fire_up_400km','built_5km']
df[cols + ['pm25']].corr(method='spearman')['pm25'].drop('pm25').sort_values().round(3).to_frame('Spearman com PM2.5')

,Spearman com PM2.5
tp,-0.451
rh,-0.414
aod_filled,-0.074
aod_dry_blh,-0.058
log_vent,-0.043
aod,0.040
built_5km,0.110
blh_pm,0.277
no2_filled,0.460
fire_up_400km,0.478
